In [ ]:
import os
import copy
import glob
import warnings # hide the warnings
from pathlib import Path

import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd                          
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


from scipy import sparse
from scipy.stats import lognorm
from scipy.stats import genextreme
from scipy.interpolate import griddata


from matplotlib.colors import BoundaryNorm
from matplotlib.colors import ListedColormap
from matplotlib.cm import ScalarMappable
import matplotlib.pyplot as plt 
from cartopy.io import shapereader   
import cartopy.crs as ccrs

from climada.hazard import Hazard
from climada.hazard import Centroids
from climada.hazard import TCTracks
from climada.hazard import TropCyclone
from climada.util.plot import plot_from_gdf

warnings.filterwarnings("ignore")

In [ ]:
###############################################################################################################################################
# below we use WRF (WRF has no synthetic)
###############################################################################################################################################

In [ ]:
files_list = sorted(glob.glob("/work/u1625133/tc-risk/PGW4K.v230926/wrfout_d01_*_48.nc")) # a list of all historical file names

In [ ]:
years = [int(os.path.basename(f).split("_")[2][:4]) for f in files_list]
num_of_years = (max(years) - min(years) + 1)

In [ ]:
histogram_x, histogram_y = np.unique(np.array(years), return_counts=True)
plt.bar(histogram_x, histogram_y)
plt.ylabel("#")
plt.xlabel("Year")

In [ ]:
###############################################################################################################################################
# below we use TC Track data (txt files)
###############################################################################################################################################

In [ ]:
tracks_list = sorted(glob.glob("/work/u1625133/tc-risk/tc_track_data/TCtrack.slp.4C_*_48.txt"))

In [ ]:
years = [int(os.path.basename(f).split("_")[1][:4]) for f in tracks_list]
num_of_years = (max(years) - min(years) + 1)

In [ ]:
histogram_x, histogram_y = np.unique(np.array(years), return_counts=True)
plt.bar(histogram_x, histogram_y)
plt.ylabel("#")
plt.xlabel("Year")

In [ ]:
###############################################################################################################################################
# below we find the overlapping time range in WRF and TC Track data
###############################################################################################################################################

In [ ]:
assert len(files_list) == len(tracks_list)

overlap_time_intersect = []


for wrf_file, track_file in zip(files_list, tracks_list):

    # Number of WRF time steps
    with xr.open_dataset(wrf_file) as ref:
        wrf_times = pd.to_datetime(ref["Times"].values.astype("U"),
                                   format="%Y-%m-%d_%H:%M:%S"
                                  )

    # Number of track time steps
    track_ref = pd.read_csv(track_file,
                            sep=r"\s+",
                            header=None,
                            comment="#"
                           )




    track_times = pd.to_datetime({"year":  track_ref.iloc[:, 1],
                                  "month": track_ref.iloc[:, 2],
                                  "day":   track_ref.iloc[:, 3],
                                  "hour":  track_ref.iloc[:, 4]}
                                 )




    overlap_times = wrf_times.intersection(track_times)

    overlap_time_intersect.append(overlap_times)

In [ ]:
def compute_category_ms(vmax_ms):  # saffir_simpson_category
    if vmax_ms < 17.49:   # 34 kt
        return -1
    if vmax_ms < 32.92:   # 64 kt
        return 0
    if vmax_ms < 42.70:   # 83 kt
        return 1
    if vmax_ms < 49.39:   # 96 kt
        return 2
    if vmax_ms < 58.13:   # 113 kt
        return 3
    if vmax_ms < 70.48:   # 137 kt
        return 4
    return 5

In [ ]:
def compute_category_kt(vmax_kt): # saffir_simpson_category
    if vmax_kt < 34:
        return -1
    if vmax_kt < 64:
        return 0
    if vmax_kt < 83:
        return 1
    if vmax_kt < 96:
        return 2
    if vmax_kt < 113:
        return 3
    if vmax_kt < 137:
        return 4
    return 5   

In [ ]:
def build_track_data(track_df, event_code, id_no, basin="WP"):
    
    time = pd.to_datetime(track_df[["year", "month", "day", "hour"]]).to_numpy(dtype="datetime64[ns]")

    ds_track = xr.Dataset(coords=dict(time=time,
                                      lat=("time", track_df["lat"].values),
                                      lon=("time", track_df["lon"].values),
                                     ),
                          
                          data_vars=dict(max_sustained_wind=("time", track_df["vmax_ms"].values * 1.94384),
                                         central_pressure=("time", track_df["pmin_hpa"].values),
                                         environmental_pressure=("time", track_df["penv_hpa"].values),
                                         radius_max_wind=("time", np.full(len(time), np.nan)), # see "def estimate_rmw(rmw, cen_pres):"
                                         basin=("time", np.array([basin] * len(time))),
                                         time_step=("time", np.ones(len(time), dtype=float)),
                                        ),

                          attrs=dict(name=event_code[6:], # e.g., "XXXXXXMEKKHALA"
                                     max_sustained_wind_unit="kt",
                                     central_pressure_unit="hPa",
                                     sid=track_df.at[0, "sid"],
                                     id_no=id_no,
                                     orig_event_flag=True,
                                     category=compute_category_ms((track_df["vmax_ms"].values).max()),
                                    ),
                          )
    return ds_track

In [ ]:
###############################################################################################################################################
# create the WRF hourly hazard object
# one event, every hour
###############################################################################################################################################

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0


with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)) # mask is of xarray.DataArray

    centroids = Centroids(
                          lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                          lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()])
###############################################################################################################################################
intensity_all_hours = []
event_name = []
for f, overlap in zip(files_list, overlap_time_intersect):
    
    event_pos_name = f.split("_")[2]  # e.g. "202002MEKKHALA"
    
    with xr.open_dataset(f) as ref:
        # intensity_all_hours
        ref = ref.swap_dims({"Time": "XTIME"}) # use "XTIME" as the dimension coordinate instead of "Time"
        u10_slice = ref["U10"].sel(XTIME=overlap)
        v10_slice = ref["V10"].sel(XTIME=overlap)

        u10_masked  = u10_slice.values[:, mask.values]
        v10_masked  = v10_slice.values[:, mask.values]

        wind_speed_hourly = np.sqrt(u10_masked**2 + v10_masked**2)        
        intensity_all_hours.append(wind_speed_hourly[1:, :]) # skip HOUR000
       
        # event_name
        event_all_hours = len(overlap)
        for event_hour_i in range(1, event_all_hours): # skip HOUR000
            event_name.append(f"{event_pos_name}_HOUR_{event_hour_i:03d}")
            

intensity = sparse.csr_matrix(np.vstack(intensity_all_hours))
assert len(event_name) == intensity.shape[0]
###############################################################################################################################################
n_hours = intensity.shape[0]
###############################################################################################################################################
frequency = np.ones(n_hours, dtype=float) # placeholder 
###############################################################################################################################################
fraction = intensity.copy()
fraction.data.fill(1)
###############################################################################################################################################
units = 'm/s'
###############################################################################################################################################
event_id = np.arange(1, n_hours + 1, dtype=int) 
###############################################################################################################################################
# date = np.array([  ])

In [ ]:
haz_hourly = TropCyclone(centroids=centroids,
                         intensity=intensity,
                         frequency=frequency,
                         event_id=event_id,
                         event_name=event_name,
                         units=units)

In [ ]:
haz_hourly.check()

In [ ]:
for name in haz_hourly.event_name:
    if "198602NANCY" in name:
        ax = haz_hourly.plot_intensity(name, vmin=0, vmax=50)
        cbar_ax = ax.get_figure().axes[-1]
        cbar_ax.set_yticks([0, 10, 20, 30, 40]) 

In [ ]:
###############################################################################################################################################
# preparation for the all_tracks_haz hazard object (without generating probabilistic synthetic events)
###############################################################################################################################################

In [ ]:
all_track_data = []

id_no = 1
for f in tracks_list:
    track_df = pd.read_csv(f,
                           sep=r"\s+",
                           header=None,
                           names=["sid","year","month","day","hour","lon","lat","vmax_ms","pmin_hpa","penv_hpa"]) # pandas.Dataframe

    filename = f.split("/")[-1] # the last element in ["", "lfs", "home", ..., "TCtrack.slp.4C_202002MEKKHALA_48.txt"] 
    event_code = filename.split("_")[1] # the second element in ["TCtrack.slp.4C", "202002MEKKHALA", "48.txt"]

    track_data = build_track_data(track_df, event_code, id_no, basin="WP")
    all_track_data.append(track_data)


    id_no += 1

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0
###############################################################################################################################################
# single_track = TCTracks(data=[all_track_data[-1]])
# single_track.plot().set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree()) # boundary the same as WRF
###############################################################################################################################################
all_tracks = TCTracks(data=all_track_data)
all_tracks.plot().set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree()) # boundary the same as WRF

In [ ]:
###############################################################################################################################################
# below we generate probabilistic/synthetic events
# for the all_tracks_haz hazard object
###############################################################################################################################################

In [ ]:
# all_tracks.equal_timestep() # must be done before sythetic generation 
# all_tracks.calc_perturbed_trajectories(nb_synth_tracks=10) # nb_synth_tracks = how many synthetic tracks is computed for every track
#                                                            # generate synthetic tracks based on directed random walk.
# all_tracks.plot()
# all_tracks.data
# all_tracks.data[-1] # the last synthetic track, notice the value of orig_event_flag and name

In [ ]:
###############################################################################################################################################
# create the all_tracks_haz hazard object (after generating probabilistic synthetic events)
###############################################################################################################################################

In [ ]:
# centroids = from WRF
with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)) # mask is of xarray.DataArray

    centroids = Centroids(
                          lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                          lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()])
###############################################################################################################################################
# other Hazard variables 
# ...
###############################################################################################################################################
# other Hazard variables  
# ...

In [ ]:
all_tracks_haz = TropCyclone.from_tracks(tracks=all_tracks, 
                                         centroids=centroids, # centroids = the same as WRF
                                         model='H1980', 
                                         model_kwargs={"gradient_to_surface_winds": 0.9},
                                         intensity_thres=0,
                                         store_windfields=True) 

In [ ]:
all_tracks_haz.check()

In [ ]:
all_tracks_haz.plot_intensity(0, vmin=0, vmax=70)
all_tracks_haz.plot_intensity(1)
all_tracks_haz.plot_intensity(2)
all_tracks_haz.plot_intensity("202002MEKKHALA", vmin=0, vmax=40)
all_tracks_haz.plot_intensity("202003BAVI", vmin=0, vmax=40)
all_tracks_haz.plot_intensity("202004ATSANI", vmin=0, vmax=40)

In [ ]:
###############################################################################################################################################
# create the all_tracks_haz_hourly hazard object (after generating probabilistic synthetic events)
# one event, every hour
# n_events -> n_hours
###############################################################################################################################################

In [ ]:
# centroids = from WRF 
with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)
           ) # mask is of xarray.DataArray

    centroids = Centroids(
                          lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                          lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()]
                         )
###############################################################################################################################################
n_hours = sum( len(overlap) - 1 for overlap in overlap_time_intersect )

n_centroids = centroids.size

intensity_all_hours = np.empty((n_hours, n_centroids), dtype=float)

row_i = 0

for event_pos in range(len(overlap_time_intersect)):
    
    event_all_hours = len(overlap_time_intersect[event_pos])
    
    for event_hour_i in range(1, event_all_hours): # skip HOUR000
        wind_vec_hour = (all_tracks_haz.windfields[event_pos][event_hour_i, :]
                         .toarray()
                         .reshape(n_centroids, 2)
                        ) # u_wind and v_wind per centroid
        
        u_wind = wind_vec_hour[:, 0] # first wind-vector component at each centroid
        v_wind = wind_vec_hour[:, 1] # second wind-vector component at each centroid
        
        wind_speed_hour = np.sqrt(u_wind**2 + v_wind**2)

        intensity_all_hours[row_i, :] = wind_speed_hour

        row_i += 1 # add one after filling each row

assert row_i == n_hours
intensity = sparse.csr_matrix(intensity_all_hours)
###############################################################################################################################################
fraction = intensity.copy()
fraction.data.fill(1)
###############################################################################################################################################
event_name = []
for f, overlap in zip(tracks_list, overlap_time_intersect):
    
    event_pos_name = Path(f).name.split("_")[1]  # e.g. "202002MEKKHALA"
    event_all_hours = len(overlap)

    for event_hour_i in range(1, event_all_hours): # skip HOUR000
        event_name.append(
            f"{event_pos_name}_HOUR_{event_hour_i:03d}"   )

assert len(event_name) == n_hours    
###############################################################################################################################################
units = 'm/s'
###############################################################################################################################################
event_id = np.arange(1, n_hours + 1, dtype=int) 
###############################################################################################################################################
n_ev = n_hours
###############################################################################################################################################
frequency = np.ones(n_hours, dtype=float) # placeholder 

In [ ]:
all_tracks_haz_hourly = TropCyclone(centroids=centroids,
                                    intensity=intensity,
                                    frequency=frequency,
                                    event_id=event_id,
                                    event_name=event_name,
                                    units=units)

In [ ]:
all_tracks_haz_hourly.check()

In [ ]:
for name in all_tracks_haz_hourly.event_name:
    if "198602NANCY" in name:
        ax = all_tracks_haz_hourly.plot_intensity(name, vmin=0, vmax=50)
        cbar_ax = ax.get_figure().axes[-1]
        cbar_ax.set_yticks([0, 10, 20, 30, 40]) 
        

In [ ]:
###############################################################################################################################################
# ERA5 anomaly for CNN 
###############################################################################################################################################

In [ ]:
wind_anom_tw = xr.open_dataset("/work/u1625133/tc-risk/ERA5/wind_anom_tw.nc")

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)
           ) # mask is of xarray.DataArray

    lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()]
    lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()]



pts_lat = xr.DataArray(lat, dims="points")
pts_lon = xr.DataArray(lon, dims="points") 




regrid = [] # regrid[0] has all hours in event 0, 
            # ...,  
            # regrid[196] has all hours in event 196
for overlap in overlap_time_intersect:

    track_times = pd.DatetimeIndex(overlap)[1:] # skip HOUR000

    wind_anom_tw_time_hourly = wind_anom_tw.sel(time=track_times)


    # sort before interpolation
    wind_anom_tw_time_hourly = wind_anom_tw_time_hourly.sortby(["latitude", "longitude"])

    # interpolate ERA5 anomaly field to WRF mask points
    r = wind_anom_tw_time_hourly.interp(latitude=pts_lat,
                                        longitude=pts_lon,
                                        method="linear")

    regrid.append(r)

In [ ]:
###############################################################################################################################################
# Terrain for CNN 
###############################################################################################################################################

In [ ]:
terr = xr.open_dataset("/work/u1625133/tc-risk/terrain/wrf_pgw_twn_grid_coords.nc")

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)) # mask is of xarray.DataArray

    lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()]
    lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()]




# terr
ter_lat = terr["XLAT"].values.ravel()
ter_lon = terr["XLONG"].values.ravel()
ter_values = terr["TER"].values.ravel()


# interpolation
# r_terrain is a 1d numpy array
r_terrain = griddata(points=(ter_lon, ter_lat), # from terr points
                     values=ter_values,
                     xi=(lon, lat), # to WRF mask points
                     method="linear")

In [ ]:
###############################################################################################################################################
# CNN 6xxx hours <-> 6xxx hours
###############################################################################################################################################

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = ( (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
             (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max) ) # mask is of (499*549)


n_hours = all_tracks_haz_hourly.intensity.shape[0] # use the input X

y_idx, x_idx = np.where(mask.values)
ny = y_idx.max() - y_idx.min() + 1
nx = x_idx.max() - x_idx.min() + 1
mask_rect = mask.isel( south_north = slice(y_idx.min(), y_idx.max() + 1),
                       west_east   = slice(x_idx.min(), x_idx.max() + 1) ) # mask_rect is of (254*235)

X_img     = np.zeros((n_hours, ny, nx))
era5_img  = np.zeros((n_hours, ny, nx))
terr_img  = np.zeros((n_hours, ny, nx))
Y_img     = np.zeros((n_hours, ny, nx))

X_img[:, mask_rect.values]    = all_tracks_haz_hourly.intensity.toarray() # ! vectorization
era5_img[:, mask_rect.values] = np.vstack([  r["sqrt_wind_speed_anomaly"].values 
                                             for r in regrid  ]) # ! vectorization
terr_img[:, mask_rect.values] = r_terrain # r_terrain is a 1d numpy array
Y_img[:, mask_rect.values]    = haz_hourly.intensity.toarray() # ! vectorization



X_cnn   = np.stack([X_img, era5_img, terr_img], axis=1) # numpy.ndarray (n_hours, channels, ny, nx)
Y_cnn   = np.stack([Y_img          ], axis=1) # numpy.ndarray (n_hours, 1, ny, nx)
res_cnn = Y_cnn - X_cnn[:, 0:1, :, :] # 0:1 selects the channel in X_cnn

In [ ]:
X_tensor   = torch.tensor(X_cnn, dtype=torch.float32)
Y_tensor   = torch.tensor(Y_cnn, dtype=torch.float32)
res_tensor = torch.tensor(res_cnn, dtype=torch.float32)

In [ ]:
torch.manual_seed(42)  # fix training, validation, test samples
                       # fix initial CNN weights

hours_per_event = torch.tensor([len(i) - 1 for i in overlap_time_intersect])
assert hours_per_event.sum() == n_hours


# shuffle EVENTS, not HOURS
event_perm = torch.randperm(len(overlap_time_intersect))

cumulative_hours = torch.cumsum(hours_per_event[event_perm], dim=0)

cut_70 = torch.argmin(torch.abs(cumulative_hours 
                                - 0.70 * n_hours)).item() + 1
cut_85 = torch.argmin(torch.abs(cumulative_hours 
                                - 0.85 * n_hours)).item() + 1

train_idx      = event_perm[:cut_70]
validation_idx = event_perm[cut_70:cut_85]
test_idx       = event_perm[cut_85:]

hour_idx_by_event = torch.split(torch.arange(n_hours),
                                hours_per_event.tolist())

train_idx      = torch.cat([hour_idx_by_event[i] for i in train_idx])
validation_idx = torch.cat([hour_idx_by_event[i] for i in validation_idx])
test_idx       = torch.cat([hour_idx_by_event[i] for i in test_idx])

In [ ]:
X_tensor_train        = X_tensor[train_idx]
X_tensor_validation   = X_tensor[validation_idx]
X_tensor_test         = X_tensor[test_idx]

res_tensor_train      = res_tensor[train_idx]
res_tensor_validation = res_tensor[validation_idx]

Y_tensor_test         = Y_tensor[test_idx]

In [ ]:
all_tensor_train      = TensorDataset(X_tensor_train, res_tensor_train)
all_tensor_validation = TensorDataset(X_tensor_validation, res_tensor_validation)

train_loader      = DataLoader(all_tensor_train, batch_size=64, shuffle=True) # shuffle at every epoch
validation_loader = DataLoader(all_tensor_validation, batch_size=64, shuffle=False) 

In [ ]:
# axis=0: n_ev
# axis=2: n_mask_points
X_mean        = X_cnn[train_idx][:, 0:3, mask_rect.values].mean(axis=(0, 2)) 
X_mean        = X_mean.reshape(1, 3, 1, 1) 
X_mean_tensor = torch.tensor(X_mean, dtype=torch.float32)

X_std         = X_cnn[train_idx][:, 0:3, mask_rect.values].std(axis=(0, 2)) 
X_std         = X_std.reshape(1, 3, 1, 1) 
X_std_tensor  = torch.tensor(X_std, dtype=torch.float32)

mask_rect_tensor = torch.tensor(mask_rect.values, dtype=torch.bool)

In [ ]:
class CNN(nn.Module): 
    def __init__(self, X_mean_tensor, X_std_tensor, mask_rect_tensor):
        super().__init__() # nn.Module.__init__(self)

        self.net    = nn.Sequential(nn.Conv2d(3, 16, kernel_size=(3, 3), stride=1, padding=1),  # input: (n_hours, channels, nlat, nlon) -> (n_hours, 16, nlat, nlon)                               
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU


                                    nn.Conv2d(16, 32, kernel_size=(3, 3), stride=1, padding=1), # (n_hours, 16, nlat, nlon) -> (n_hours, 32, nlat, nlon)
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU


                                    nn.Conv2d(32, 32, kernel_size=(3, 3), stride=1, padding=1), # (n_hours, 32, nlat, nlon) -> (n_hours, 32, nlat, nlon)
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU


                                    nn.Conv2d(32, 32, kernel_size=(3, 3), stride=1, padding=1), # (n_hours, 32, nlat, nlon) -> (n_hours, 32, nlat, nlon)
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU


                                    nn.Conv2d(32, 16, kernel_size=(3, 3), stride=1, padding=1), # (n_hours, 32, nlat, nlon) -> (n_hours, 16, nlat, nlon)
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU


                                    nn.Conv2d(16, 1, kernel_size=(3, 3), stride=1, padding=1) # output: res_pred 
                                                                                              # output: (n_hours, 1, nlat, nlon)
                                                                                              # Y_pred = X + res_pred
                                    )

        self.register_buffer("X_mean_tensor", X_mean_tensor)
        self.register_buffer("X_std_tensor", X_std_tensor)
        self.register_buffer("mask_rect_tensor", mask_rect_tensor)


    def forward(self, X):
        
        normalized_X = (X - self.X_mean_tensor) / self.X_std_tensor # X is of torch.tensor
        
        normalized_X[:, :, ~self.mask_rect_tensor] = 0.0     
        
        return self.net(normalized_X) # meaning nn.Sequential(...)(normalized_X)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN(X_mean_tensor, X_std_tensor, mask_rect_tensor).to(device)

loss_func = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 200



for epoch in range(n_epochs):
    
    model.train()

    train_loss_all_batches = 0.0
    train_samples_all_batches = 0

    
    for x, r in train_loader: # mini batches per epoch
        
        optimizer.zero_grad() # clear gradients from previous batch


        X_tensor_train_batch = x.to(device)
        r_pred_batch = model(X_tensor_train_batch) # model.forward(X_tensor[train_idx])

        r_true_batch = r.to(device)
        
        train_loss_per_batch = loss_func(r_pred_batch[:, :, model.mask_rect_tensor],  
                                         r_true_batch[:, :, model.mask_rect_tensor])

        # learning weights
        train_loss_per_batch.backward() # computes the new gradients
        optimizer.step() # update model weights


        train_loss_all_batches += train_loss_per_batch.item() * x.size(0)
        train_samples_all_batches += x.size(0)

    mean_train_loss_per_epoch = train_loss_all_batches / train_samples_all_batches
   
    if epoch % 10 == 0: 
       # validation
       model.eval()

       validation_loss_all_batches = 0.0
       validation_samples_all_batches = 0
        
       with torch.no_grad():
           for x, r in validation_loader: 
               X_tensor_validation_batch = x.to(device)
               r_validation_pred_batch = model(X_tensor_validation_batch)
               
               r_validation_true_batch = r.to(device)
               
               validation_loss_per_batch = loss_func(r_validation_pred_batch[:, :, model.mask_rect_tensor], 
                                                     r_validation_true_batch[:, :, model.mask_rect_tensor])
               
               validation_loss_all_batches += validation_loss_per_batch.item() * x.size(0)
               validation_samples_all_batches += x.size(0)
           
           mean_validation_loss_this_epoch =  validation_loss_all_batches / validation_samples_all_batches


        

       print(f"epoch {epoch}, "
             f"train mse: {mean_train_loss_per_epoch:.6f}, "
             f"validation mse: {mean_validation_loss_this_epoch:.6f}")

In [ ]:
model.eval()
with torch.no_grad():
   
    # model predicts residual
    X_tensor_test_gpu = X_tensor_test.to(device)
    r_pred_test_gpu = model(X_tensor_test_gpu)
    # final corrected wind
    Y_pred_test_gpu = X_tensor_test_gpu[:, 0:1, :, :] + r_pred_test_gpu


    Y_tensor_test_gpu = Y_tensor_test.to(device) # only 1 channel
    
    mae = torch.mean( torch.abs(Y_tensor_test_gpu[:, :, model.mask_rect_tensor] - 
                                Y_pred_test_gpu[:, :, model.mask_rect_tensor]) )
    
    rmse = torch.sqrt( torch.mean((Y_tensor_test_gpu[:, :, model.mask_rect_tensor] -
                                   Y_pred_test_gpu[:, :, model.mask_rect_tensor]) ** 2) )

    old_mae = torch.mean( torch.abs(Y_tensor_test_gpu[:, :, model.mask_rect_tensor] - 
                                    X_tensor_test_gpu[:, 0:1, model.mask_rect_tensor]) )
    
    old_rmse = torch.sqrt (torch.mean((Y_tensor_test_gpu[:, :, model.mask_rect_tensor] - 
                                       X_tensor_test_gpu[:, 0:1, model.mask_rect_tensor]) ** 2) )

In [ ]:
print("old MAE:", old_mae.item())
print("old RMSE:", old_rmse.item())
print("MAE:", mae.item())
print("RMSE:", rmse.item())

In [ ]:
###############################################################################################################################################
# CNN 6xxx hours <-> 6xxx hours
# TESTING VISUALIZATION
###############################################################################################################################################

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = ( (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
             (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max) ) # mask is of xarray.DataArray

    centroids = Centroids( lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                           lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()] )

###############################################################################################################################################
units = 'm/s'
###############################################################################################################################################
Y_pred_test_n_ev = len(Y_pred_test_gpu) # gives the first dimension
###############################################################################################################################################
Y_pred_test_event_id = test_idx.numpy()+1    
###############################################################################################################################################
# from (n_events, 1, nlat, nlon) to (n_events, nlat, nlon)
Y_pred_test = Y_pred_test_gpu.squeeze(1).cpu().numpy() 

# from (n_events, nlat, nlon) to (n_events, n_centroids)
flat_Y_pred_test = Y_pred_test[:, mask_rect.values] 

# # turn negative to positive for the compressed sparse matrix
# flat_Y_pred_test = np.maximum(flat_Y_pred_test, 0)

# convert to intensity sparse matrix
Y_pred_test_intensity = sparse.csr_matrix(flat_Y_pred_test.astype(np.float32))
###############################################################################################################################################
Y_pred_test_frequency =  np.ones(Y_pred_test_n_ev) # placeholder for Hazard object  
###############################################################################################################################################
Y_pred_test_fraction = Y_pred_test_intensity.copy()
Y_pred_test_fraction.data.fill(1)
###############################################################################################################################################
Y_pred_test_event_name = [all_tracks_haz_hourly.event_name[i] for i in test_idx.numpy()] 
###############################################################################################################################################
# date = np.array([  ])
###############################################################################################################################################
# orig = np.ones(n_ev, bool)

In [ ]:
Y_pred_test_haz = Hazard(intensity=Y_pred_test_intensity,
                         fraction=Y_pred_test_fraction,
                         centroids=centroids,  
                         units=units,
                         frequency=Y_pred_test_frequency,  
                         event_id=Y_pred_test_event_id,  
                         event_name=Y_pred_test_event_name)

In [ ]:
Y_pred_test_haz.check()

In [ ]:
Y_pred_test_haz.event_name

In [ ]:
for name in Y_pred_test_haz.event_name:
    if "198602NANCY" in name:
        ax = Y_pred_test_haz.plot_intensity(event=name, vmin=0, vmax=50)
        cbar_ax = ax.get_figure().axes[-1]
        cbar_ax.set_yticks([0, 10, 20, 30, 40]) 

        


